# Υπολογιστική Όραση

Σκοπός της εργασίας είναι η υλοποίηση μιας εφαρμογής σε Python/OpenCV η οποία λαμβάνει εικόνα από κάμερα σε πραγματικό χρόνο και εφαρμόζει μεθόδους ανίχνευσης ακμών.

Οι μέθοδοι που θα χρησιμοποιηθούν είναι:

- Sobel
- Canny
- Laplacian

Μετά την ανίχνευση ακμών, η εφαρμογή θα βρίσκει τα περιγράμματα των αντικειμένων, δηλαδή τα contours.

Στη συνέχεια, με βάση απλά γεωμετρικά χαρακτηριστικά όπως το εμβαδόν, το bounding box και το aspect ratio, θα γίνεται απλή κατηγοριοποίηση αντικειμένων όπως:

- μπουκάλι
- τετράδιο
- κουτί

Η εφαρμογή δεν χρησιμοποιεί τεχνητή νοημοσύνη ή εκπαιδευμένο μοντέλο. Η αναγνώριση γίνεται με απλούς κανόνες, δηλαδή rule-based classification.

## Εγκατάσταση βιβλιοθηκών

Για την υλοποίηση χρησιμοποιούμε τις βιβλιοθήκες:

- OpenCV
- NumPy

Η OpenCV χρησιμοποιείται για την επεξεργασία εικόνας, την κάμερα, τα φίλτρα ακμών και τα contours.

Η NumPy χρησιμοποιείται για αριθμητικές πράξεις πάνω στις εικόνες.

In [289]:
%pip install opencv-python numpy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## Εισαγωγή βιβλιοθηκών

Σε αυτό το σημείο εισάγουμε τις απαραίτητες βιβλιοθήκες.

Το `cv2` είναι η OpenCV.

Το `numpy` χρησιμοποιείται για πράξεις πάνω σε πίνακες.

Το `time` χρησιμοποιείται για τον υπολογισμό των FPS.

In [290]:
import cv2
import numpy as np
import time

## Βασικές ρυθμίσεις

Ορίζουμε κάποιες βασικές σταθερές για το πρόγραμμα.

Το `CAMERA_INDEX = 0` σημαίνει ότι χρησιμοποιούμε την πρώτη κάμερα του υπολογιστή.

Το `FRAME_WIDTH = 800` σημαίνει ότι κάθε frame θα γίνεται resize ώστε να έχει πλάτος 800 pixels.

Το `MIN_CONTOUR_AREA` χρησιμοποιείται για να αγνοούμε πολύ μικρά contours, επειδή συνήθως είναι θόρυβος.

Το `MAX_OBJECTS_TO_DRAW` περιορίζει πόσα αντικείμενα θα σχεδιάζονται στην εικόνα.

In [291]:
CAMERA_INDEX = 0
FRAME_WIDTH = 800

MIN_CONTOUR_AREA = 2500
MIN_BOX_AREA = 7000
MAX_OBJECTS_TO_DRAW = 6

SHOW_UNKNOWN_OBJECTS = False

## Resize του frame

Η κάμερα μπορεί να επιστρέφει εικόνες σε μεγάλο μέγεθος.

Για να τρέχει πιο γρήγορα η εφαρμογή, κάνουμε resize το frame σε σταθερό πλάτος.

Κρατάμε όμως το σωστό aspect ratio, ώστε να μη χαλάσει η εικόνα.

In [292]:
def resize_frame(frame, width=FRAME_WIDTH):
    """
    Κάνει resize το frame σε συγκεκριμένο πλάτος,
    κρατώντας σωστό aspect ratio.
    """
    
    height, original_width = frame.shape[:2]
    
    scale = width / original_width
    new_height = int(height * scale)
    
    resized = cv2.resize(frame, (width, new_height))
    
    return resized

## Προεπεξεργασία εικόνας

Πριν εφαρμόσουμε edge detection, κάνουμε δύο βασικά βήματα:

1. Μετατροπή της εικόνας σε grayscale.
2. Εφαρμογή Gaussian Blur.

Η μετατροπή σε grayscale γίνεται επειδή οι αλγόριθμοι ακμών δουλεύουν πάνω στις αλλαγές φωτεινότητας.

Το Gaussian Blur μειώνει τον θόρυβο, ώστε να μην ανιχνεύονται πολλές ψεύτικες ακμές.

In [293]:
def preprocess_frame(frame):
    """
    Μετατρέπει το frame σε grayscale και εφαρμόζει Gaussian Blur.
    """
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    return blurred

## Κανονικοποίηση εικόνας

Οι μέθοδοι Sobel και Laplacian μπορεί να παράγουν τιμές που δεν είναι απευθείας κατάλληλες για εμφάνιση ως εικόνα.

Για αυτό, δημιουργούμε μια βοηθητική συνάρτηση που μετατρέπει το αποτέλεσμα σε εικόνα με τιμές από 0 έως 255.

In [294]:
def normalize_to_uint8(image):
    """
    Μετατρέπει μία εικόνα σε μορφή uint8 με τιμές από 0 έως 255.
    """
    
    normalized = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
    
    normalized = normalized.astype(np.uint8)
    
    return normalized

## Εφαρμογή Edge Detection

Σε αυτή τη συνάρτηση εφαρμόζουμε μία από τις τρεις μεθόδους:

- Canny
- Sobel
- Laplacian

Η επιλογή της μεθόδου γίνεται με τη μεταβλητή `method`.

In [295]:
def apply_edge_detection(gray_blurred, method):
    """
    Εφαρμόζει edge detection ανάλογα με τη μέθοδο που επιλέγουμε.
    
    method:
    - "canny"
    - "sobel"
    - "laplacian"
    """
    
    if method == "canny":
        edges = cv2.Canny(gray_blurred, 60, 160)
        return edges
    
    
    elif method == "sobel":
        # Sobel στον άξονα x
        grad_x = cv2.Sobel(gray_blurred, cv2.CV_64F, 1, 0, ksize=3)
        
        # Sobel στον άξονα y
        grad_y = cv2.Sobel(gray_blurred, cv2.CV_64F, 0, 1, ksize=3)
        
        # Συνδυασμός των δύο gradients
        magnitude = cv2.magnitude(grad_x, grad_y)
        
        # Μετατροπή σε εικόνα 0-255
        magnitude = normalize_to_uint8(magnitude)
        
        # Μετατροπή σε ασπρόμαυρη δυαδική εικόνα
        _, edges = cv2.threshold(
            magnitude,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        
        return edges
    
    
    elif method == "laplacian":
        # Εφαρμογή Laplacian
        lap = cv2.Laplacian(gray_blurred, cv2.CV_64F, ksize=3)
        
        # Παίρνουμε απόλυτες τιμές
        lap = np.absolute(lap)
        
        # Μετατροπή σε εικόνα 0-255
        lap = normalize_to_uint8(lap)
        
        # Μετατροπή σε δυαδική εικόνα
        _, edges = cv2.threshold(
            lap,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        
        return edges
    
    
    else:
        raise ValueError("Άγνωστη μέθοδος edge detection")

## Καθαρισμός ακμών

Μετά το edge detection, οι ακμές μπορεί να έχουν μικρά κενά.

Για να βγουν καλύτερα contours, χρησιμοποιούμε μορφολογικές πράξεις.

Το `MORPH_CLOSE` βοηθάει να κλείσουν μικρά κενά στις γραμμές.

Το `dilate` παχαίνει λίγο τις γραμμές, ώστε να ενώνονται καλύτερα τα περιγράμματα.

In [296]:
def clean_edges(edges):
    """
    Καθαρίζει τις ακμές χωρίς να ενώνει υπερβολικά άσχετα σημεία.
    """

    kernel = np.ones((3, 3), np.uint8)

    # Κλείνει μικρά κενά, αλλά όχι πολύ επιθετικά
    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=1)

    return closed

## Εξαγωγή χαρακτηριστικών από contour

Για κάθε contour υπολογίζουμε βασικά χαρακτηριστικά.

Τα πιο σημαντικά είναι:

- area: εμβαδόν
- perimeter: περίμετρος
- bounding box: ορθογώνιο που περικλείει το αντικείμενο
- width και height
- aspect ratio
- αριθμός κορυφών

Το aspect ratio υπολογίζεται ως:

height / width

Αυτό μας βοηθάει να ξεχωρίσουμε απλά σχήματα.

Για παράδειγμα:

- Ένα μπουκάλι είναι συνήθως ψηλό και στενό.
- Ένα τετράδιο είναι συνήθως πλατύ ορθογώνιο.
- Ένα κουτί είναι πιο συμπαγές ορθογώνιο ή τετράγωνο.

In [297]:
def extract_features(contour):
    """
    Υπολογίζει γεωμετρικά χαρακτηριστικά από ένα contour.
    """

    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)

    x, y, w, h = cv2.boundingRect(contour)

    box_area = w * h

    if w != 0:
        aspect_ratio = h / float(w)
    else:
        aspect_ratio = 0

    if box_area != 0:
        extent = area / float(box_area)
    else:
        extent = 0

    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)

    if hull_area != 0:
        solidity = area / float(hull_area)
    else:
        solidity = 0

    epsilon = 0.04 * perimeter
    approx = cv2.approxPolyDP(contour, epsilon, True)
    vertices = len(approx)

    features = {
        "area": area,
        "perimeter": perimeter,
        "x": x,
        "y": y,
        "w": w,
        "h": h,
        "box_area": box_area,
        "vertices": vertices,
        "aspect_ratio": aspect_ratio,
        "extent": extent,
        "solidity": solidity
    }

    return features

## Απλή αναγνώριση αντικειμένων

Σε αυτό το σημείο δεν χρησιμοποιούμε machine learning.

Χρησιμοποιούμε απλούς κανόνες.

Η λογική είναι:

- Αν το αντικείμενο είναι ψηλό και στενό, το θεωρούμε πιθανό μπουκάλι.
- Αν έχει περίπου 4 κορυφές και είναι πλατύ, το θεωρούμε πιθανό τετράδιο.
- Αν έχει περίπου 4 κορυφές και είναι πιο συμπαγές, το θεωρούμε πιθανό κουτί.

Αυτό ονομάζεται rule-based classification.

Είναι σημαντικό να αναφέρουμε στην εργασία ότι αυτή η μέθοδος λειτουργεί σε ελεγχόμενο περιβάλλον και όχι σε οποιαδήποτε εικόνα.

In [298]:
def classify_object(features):
    """
    Απλή rule-based κατηγοριοποίηση αντικειμένων.
    """

    area = features["area"]
    w = features["w"]
    h = features["h"]
    box_area = features["box_area"]
    vertices = features["vertices"]
    aspect_ratio = features["aspect_ratio"]
    extent = features["extent"]
    solidity = features["solidity"]

    # Phone: κάθετο ορθογώνιο, σχετικά γεμάτο και συμπαγές
    if (
        1.45 <= aspect_ratio <= 2.7
        and box_area > 10000
        and 4 <= vertices <= 10
        and extent > 0.35
        and solidity > 0.60
    ):
        label = "Phone"
        color = (255, 255, 0)

    # Bottle: πιο ψηλό και στενό από κινητό
    elif (
        aspect_ratio > 2.2
        and box_area > 9000
        and extent > 0.20
        and solidity > 0.45
    ):
        label = "Bottle"
        color = (0, 255, 0)

    # Notebook: πλατύ μεγάλο ορθογώνιο
    elif (
        w > h * 1.45
        and box_area > 18000
        and 4 <= vertices <= 10
        and extent > 0.35
        and solidity > 0.60
    ):
        label = "Notebook"
        color = (255, 0, 0)

    # Sunglasses: φαρδύ και χαμηλό αντικείμενο, αλλά όχι τεράστιο
    elif (
        w > h * 1.8
        and 4000 < box_area < 30000
        and extent > 0.18
        and solidity > 0.35
    ):
        label = "Sunglasses"
        color = (255, 0, 255)

    # Box: πιο ισορροπημένο ορθογώνιο/τετράγωνο
    elif (
        0.65 <= aspect_ratio <= 1.55
        and box_area > 14000
        and 4 <= vertices <= 10
        and extent > 0.35
        and solidity > 0.60
    ):
        label = "Box"
        color = (0, 165, 255)

    else:
        label = "Unknown"
        color = (255, 255, 255)

    return label, color

## Εύρεση και φιλτράρισμα contours

Η OpenCV βρίσκει όλα τα contours στην εικόνα.

Όμως πολλά από αυτά μπορεί να είναι θόρυβος.

Για αυτό κάνουμε φιλτράρισμα:

- Αγνοούμε contours με πολύ μικρό εμβαδόν.
- Αγνοούμε πολύ μικρά bounding boxes.
- Κρατάμε μόνο τα μεγαλύτερα contours.

In [299]:
def find_valid_contours(edges, frame_shape):
    """
    Βρίσκει contours και κρατάει μόνο όσα μοιάζουν με πραγματικά αντικείμενα.
    """

    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    frame_height, frame_width = frame_shape[:2]
    frame_area = frame_height * frame_width

    valid_contours = []

    for contour in contours:
        features = extract_features(contour)

        area = features["area"]
        x = features["x"]
        y = features["y"]
        w = features["w"]
        h = features["h"]
        box_area = features["box_area"]
        aspect_ratio = features["aspect_ratio"]
        extent = features["extent"]
        solidity = features["solidity"]

        # Πετάμε μικρά contours
        if area < MIN_CONTOUR_AREA:
            continue

        # Πετάμε μικρά bounding boxes
        if box_area < MIN_BOX_AREA:
            continue

        # Πετάμε υπερβολικά μεγάλα contours, π.χ. τραπέζι/φόντο
        if box_area > 0.45 * frame_area:
            continue

        # Πετάμε πολύ μικρές διαστάσεις
        if w < 45 or h < 45:
            continue

        # Πετάμε contours που ακουμπάνε στα όρια της εικόνας
        margin = 8
        if x <= margin or y <= margin:
            continue

        if x + w >= frame_width - margin or y + h >= frame_height - margin:
            continue

        # Πετάμε υπερβολικά λεπτές γραμμές
        if aspect_ratio < 0.15 or aspect_ratio > 6.0:
            continue

        # Πετάμε πολύ άδεια/σπασμένα contours
        if extent < 0.18:
            continue

        # Πετάμε πολύ ακανόνιστα contours
        if solidity < 0.45:
            continue

        valid_contours.append(contour)

    valid_contours = sorted(
        valid_contours,
        key=lambda c: cv2.boundingRect(c)[2] * cv2.boundingRect(c)[3],
        reverse=True
    )

    return valid_contours[:MAX_OBJECTS_TO_DRAW]

## Σχεδίαση αντικειμένου πάνω στο frame

Για κάθε αντικείμενο σχεδιάζουμε:

- το contour
- το bounding box
- την ετικέτα του αντικειμένου
- το εμβαδόν
- τον αριθμό κορυφών

In [300]:
def draw_object_info(frame, contour, features, label, color):
    """
    Σχεδιάζει το contour, το bounding box και την ετικέτα του αντικειμένου.
    """
    
    x = features["x"]
    y = features["y"]
    w = features["w"]
    h = features["h"]
    area = features["area"]
    vertices = features["vertices"]
    
    # Σχεδίαση contour
    cv2.drawContours(frame, [contour], -1, color, 2)
    
    # Σχεδίαση bounding box
    cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
    
    # Κείμενο που θα εμφανίζεται
    text = f"{label} | AR:{features['aspect_ratio']:.2f} | E:{features['extent']:.2f}"
    
    cv2.putText(
        frame,
        text,
        (x, max(y - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        color,
        2
    )

## Εμφάνιση πληροφοριών στο παράθυρο

Στο πάνω μέρος του frame εμφανίζουμε:

- τη μέθοδο edge detection
- τα FPS
- τον αριθμό των contours
- τα πλήκτρα χειρισμού

In [301]:
def draw_header(frame, method, fps, contour_count):
    """
    Εμφανίζει πληροφορίες στο πάνω μέρος του frame.
    """
    
    text1 = f"Method: {method.upper()} | FPS: {fps:.1f} | Contours: {contour_count}"
    text2 = "Keys: 1=Canny, 2=Sobel, 3=Laplacian, q/ESC=Exit"
    
    cv2.putText(
        frame,
        text1,
        (15, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )
    
    cv2.putText(
        frame,
        text2,
        (15, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 255, 255),
        2
    )

In [302]:
STOP_BUTTON_BOX = None
app_state = {
    "stop": False
}


def draw_stop_button(frame):
    """
    Σχεδιάζει κουμπί τερματισμού πάνω δεξιά στο frame.
    """

    global STOP_BUTTON_BOX

    button_width = 160
    button_height = 45
    margin = 15

    x1 = frame.shape[1] - button_width - margin
    y1 = margin
    x2 = x1 + button_width
    y2 = y1 + button_height

    STOP_BUTTON_BOX = (x1, y1, x2, y2)

    # Κόκκινο γεμάτο κουμπί
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), -1)

    # Άσπρο περίγραμμα
    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 255), 2)

    # Κείμενο κουμπιού
    cv2.putText(
        frame,
        "TERMINATE",
        (x1 + 15, y1 + 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )


def mouse_callback(event, x, y, flags, param):
    """
    Ελέγχει αν ο χρήστης έκανε click πάνω στο κουμπί TERMINATE.
    """

    global STOP_BUTTON_BOX

    if event == cv2.EVENT_LBUTTONDOWN and STOP_BUTTON_BOX is not None:
        x1, y1, x2, y2 = STOP_BUTTON_BOX

        if x1 <= x <= x2 and y1 <= y <= y2:
            app_state["stop"] = True
            print("Πατήθηκε το κουμπί TERMINATE.")

## Κύρια εφαρμογή

Αυτή είναι η βασική συνάρτηση του προγράμματος.

Η ροή της εφαρμογής είναι:

1. Άνοιγμα κάμερας.
2. Λήψη frame.
3. Resize.
4. Grayscale και blur.
5. Edge detection.
6. Καθαρισμός ακμών.
7. Εύρεση contours.
8. Εξαγωγή χαρακτηριστικών.
9. Κατηγοριοποίηση αντικειμένων.
10. Σχεδίαση αποτελεσμάτων.
11. Υπολογισμός FPS.

Με τα πλήκτρα:

- 1 επιλέγουμε Canny.
- 2 επιλέγουμε Sobel.
- 3 επιλέγουμε Laplacian.
- q ή ESC κλείνουμε το πρόγραμμα.

In [303]:
def run_realtime_app():
    """
    Εκτελεί την εφαρμογή real-time video.
    Η εφαρμογή κλείνει είτε με:
    - click στο κουμπί TERMINATE
    - q
    - ESC
    - κλείσιμο του παραθύρου
    """

    app_state["stop"] = False

    window_name = "Real-Time Object Contour Detection"
    edges_window_name = "Edges"

    cap = cv2.VideoCapture(CAMERA_INDEX)

    if not cap.isOpened():
        print("Δεν μπόρεσε να ανοίξει η κάμερα.")
        return

    method = "canny"
    previous_time = time.time()

    print("Η εφαρμογή ξεκίνησε.")
    print("Πάτησε 1 για Canny.")
    print("Πάτησε 2 για Sobel.")
    print("Πάτησε 3 για Laplacian.")
    print("Πάτησε το κουμπί TERMINATE ή q για έξοδο.")

    try:
        cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
        cv2.namedWindow(edges_window_name, cv2.WINDOW_NORMAL)

        # Ενεργοποίηση mouse click στο βασικό παράθυρο
        cv2.setMouseCallback(window_name, mouse_callback)

        while not app_state["stop"]:
            ret, frame = cap.read()

            if not ret:
                print("Δεν μπόρεσε να διαβαστεί frame από την κάμερα.")
                break

            # Resize frame
            frame = resize_frame(frame)

            # Αντιγραφή για να σχεδιάσουμε πάνω σε αυτή
            output = frame.copy()

            # Προεπεξεργασία
            gray_blurred = preprocess_frame(frame)

            # Edge detection
            edges = apply_edge_detection(gray_blurred, method)

            # Καθαρισμός ακμών
            cleaned_edges = clean_edges(edges)

            # Εύρεση contours
            contours = find_valid_contours(cleaned_edges, frame.shape)

            # Για κάθε contour βρίσκουμε χαρακτηριστικά και το ταξινομούμε
            for contour in contours:
                features = extract_features(contour)

                label, color = classify_object(features)

                if label == "Unknown" and not SHOW_UNKNOWN_OBJECTS:
                    continue

                draw_object_info(output, contour, features, label, color)

            # Υπολογισμός FPS
            current_time = time.time()
            elapsed_time = current_time - previous_time

            if elapsed_time > 0:
                fps = 1 / elapsed_time
            else:
                fps = 0

            previous_time = current_time

            # Εμφάνιση πληροφοριών
            draw_header(output, method, fps, len(contours))

            # Σχεδίαση κουμπιού τερματισμού
            draw_stop_button(output)

            # Παράθυρα OpenCV
            cv2.imshow(window_name, output)
            cv2.imshow(edges_window_name, cleaned_edges)

            # Έλεγχος πλήκτρων
            key = cv2.waitKey(10) & 0xFF

            if key == ord("1"):
                method = "canny"
                print("Μέθοδος: Canny")

            elif key == ord("2"):
                method = "sobel"
                print("Μέθοδος: Sobel")

            elif key == ord("3"):
                method = "laplacian"
                print("Μέθοδος: Laplacian")

            elif key == ord("q") or key == 27:
                app_state["stop"] = True
                print("Ζητήθηκε έξοδος από το πληκτρολόγιο.")

            # Αν κλείσεις το παράθυρο με Χ, να σταματάει και το loop
            if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                app_state["stop"] = True
                print("Το παράθυρο έκλεισε.")

    finally:
        cap.release()
        cv2.destroyAllWindows()

        # Βοηθάει σε Mac/Jupyter να κλείσουν σωστά τα OpenCV windows
        for _ in range(5):
            cv2.waitKey(1)

        print("Η κάμερα απελευθερώθηκε και τα παράθυρα έκλεισαν.")

## Εκτέλεση εφαρμογής

Τρέχουμε το παρακάτω cell για να ξεκινήσει η εφαρμογή.

Η κάμερα θα ανοίξει σε ξεχωριστό παράθυρο OpenCV.

Για να κλείσει η εφαρμογή, πατάμε `q` ή `ESC` πάνω στο παράθυρο της OpenCV.

In [304]:
run_realtime_app()

Η εφαρμογή ξεκίνησε.
Πάτησε 1 για Canny.
Πάτησε 2 για Sobel.
Πάτησε 3 για Laplacian.
Πάτησε το κουμπί TERMINATE ή q για έξοδο.
Πατήθηκε το κουμπί TERMINATE.
Η κάμερα απελευθερώθηκε και τα παράθυρα έκλεισαν.


## Πειραματική σύγκριση

Κατά τη διάρκεια των δοκιμών, συγκρίνουμε τις τρεις μεθόδους:

| Μέθοδος | Ποιότητα ακμών | Καθαρότητα contours | FPS | Παρατηρήσεις |
|---|---|---|---|---|
| Canny | | | | |
| Sobel | | | | |
| Laplacian | | | | |

Σκοπός δεν είναι μόνο να δείξουμε ότι λειτουργεί ο κώδικας, αλλά να συγκρίνουμε ποια μέθοδος είναι καλύτερη για την εξαγωγή contours σε πραγματικό χρόνο.

## Συμπέρασμα

Η εφαρμογή υλοποιεί μια απλή μέθοδο υπολογιστικής όρασης για real-time ανίχνευση ακμών και εξαγωγή περιγραμμάτων αντικειμένων.

Μετά την εφαρμογή των φίλτρων Sobel, Canny και Laplacian, βρίσκονται contours και υπολογίζονται γεωμετρικά χαρακτηριστικά.

Με βάση αυτά τα χαρακτηριστικά εφαρμόζεται απλή rule-based κατηγοριοποίηση αντικειμένων.

Η μέθοδος αυτή δεν αποτελεί γενικό σύστημα αναγνώρισης αντικειμένων, αλλά λειτουργεί σε ελεγχόμενο περιβάλλον και δείχνει πώς μπορούν να χρησιμοποιηθούν οι κλασικές τεχνικές υπολογιστικής όρασης για την ανίχνευση και βασική ταξινόμηση αντικειμένων.